# 01 — Load and Explore Processed Visium Data

This notebook loads a previously processed human IPF Visium `AnnData` object and inspects the spatial, transcriptional, and precomputed latent representations that will be reused throughout the project.

The goal is **downstream spatial analysis**, not reproduction of the original preprocessing pipeline. We therefore reuse the existing scVI representation, UMAP coordinates, cell2location-derived cell abundances, pathway activity scores, NMF factors, and NMF-defined tissue niches.

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white")

/opt/anaconda3/envs/demo/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/var/folders/wh/ws82kl5942j9c08bkg1vwsr80000gn/T/ipykernel_39681/2912275998.py:7: FutureWarning: The method set_figure_params is deprecated and will be removed in the future. Use :func:`scanpy.set_figure_params` instead
  sc.settings.set_figure_params(dpi=100, facecolor="white")


## Load the processed AnnData object

In [2]:
# Update this path if your local file is stored elsewhere.
adata = sc.read_h5ad("../data/processed_ipf_visium.h5ad")
adata

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '../data/processed_ipf_visium.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

## Dataset overview

The processed object contains spot-level metadata, patient/sample identifiers, precomputed latent representations, spatial coordinates, cell-type abundance estimates, pathway activities, and NMF-derived niche annotations.

In [ ]:
print(f"Number of spots: {adata.n_obs:,}")
print(f"Number of genes: {adata.n_vars:,}")

print("\nSamples:")
print(adata.obs["sample"].value_counts())

print("\nTreatment:")
print(adata.obs["treatment"].value_counts())

print("\nPatients:")
print(adata.obs["patient"].value_counts())

print("\nobsm keys:")
print(list(adata.obsm.keys()))

print("\nlayers:")
print(list(adata.layers.keys()))

print("\nuns keys:")
print(list(adata.uns.keys()))

In [ ]:
sample_summary = (
    adata.obs
    .groupby(["sample", "patient", "treatment"], observed=True)
    .size()
    .reset_index(name="n_spots")
)

sample_summary

## Derived cell-type, pathway, and NMF features

The object already contains 37 cell2location-derived cell-population abundance features, 14 pathway scores, precomputed NMF factor columns, and a final `Niche_NMF` label.

In [ ]:
cell_types = [
    "AT0", "AT1", "AT2",
    "Aberrant basaloid",
    "Adventitial fibroblast",
    "Alveolar fibroblast",
    "Artery",
    "B/Plasma",
    "Basal",
    "Basophil/Mast",
    "Bronchial Vessel",
    "Capillary",
    "Capillary Aerocyte",
    "Ciliated",
    "Ciliated SFTPB+/SCGB1A1+",
    "Dendritic",
    "Ionocyte",
    "Lymphatic",
    "Macrophage C1Q hi",
    "Macrophage CHI3L1+/CD9 hi/",
    "Macrophage FABP4+",
    "Macrophage IL1B+",
    "Macrophage LYVE1+",
    "Macrophage RETN+/VCAN+",
    "Mesothelial",
    "Monocyte",
    "Mucous",
    "Myofibroblast",
    "NK",
    "Peribronchial fibroblast",
    "Pericyte",
    "Smooth Muscle",
    "T cell",
    "TB-SC",
    "Vein",
    "pDC",
    "preTB-SC/RAS"
]

pathways = [
    "Androgen",
    "EGFRsignaling",
    "Estrogen",
    "Hypoxia",
    "JAK-STAT",
    "MAPK",
    "NFkB",
    "p53",
    "PI3K",
    "TGFb",
    "TNFa",
    "Trail",
    "VEGF",
    "WNT"
]

nmf_factor_cols = [
    c for c in adata.obs.columns
    if c.startswith("mean_nUMI_factorsfact_")
]

print("Cell types:", len(cell_types))
print("Pathways:", len(pathways))
print("NMF factor columns:", len(nmf_factor_cols))
print("Niche labels:", adata.obs["Niche_NMF"].nunique())

## Precomputed representations

- `X_scVI`: scVI latent representation
- `X_umap`: precomputed UMAP coordinates
- `spatial`: Visium tissue coordinates
- `q05_cell_abundance_w_sf`: cell2location-derived abundance estimates
- `mlm_estimate`: pathway activity estimates

These representations are reused rather than recomputed.

In [ ]:
for key in [
    "X_scVI",
    "X_umap",
    "spatial",
    "q05_cell_abundance_w_sf",
    "mlm_estimate"
]:
    if key in adata.obsm:
        print(f"{key}: {adata.obsm[key].shape}")

## Explore the existing UMAP embedding

The UMAP and Leiden clustering are reused from the processed object; they are not recomputed in this demo.

In [ ]:
sc.pl.umap(
    adata,
    color=["treatment", "Niche_NMF"],
    wspace=0.4
)

In [ ]:
sc.pl.umap(
    adata,
    color="leiden_25",
    legend_loc="on data",
    legend_fontsize=6
)

## Inspect one representative IPF section

In [ ]:
sample = "90_A1_H237762_IPF_processed_CM"

adata_sample = adata[
    adata.obs["sample"] == sample
].copy()

adata_sample

In [ ]:
print("Spatial library IDs:")
print(list(adata.uns["spatial"].keys()))

## Spatial distribution of NMF-defined tissue niches

`Niche_NMF` summarizes spatially organized multicellular microenvironments derived from cell2location-based cell abundance profiles. Later notebooks focus on the `Fibrotic` niche.

In [ ]:
# If the spatial library ID differs from the sample name,
# replace library_id below with the matching key printed above.
sc.pl.spatial(
    adata_sample,
    color="Niche_NMF",
    library_id=sample,
    spot_size=1.3
)

## Representative spatial gene-expression patterns

A small set of fibrosis-associated and alveolar markers is shown as a basic biological sanity check. Systematic spatial gene/pathway analyses are reserved for later notebooks.

In [ ]:
genes = [
    "COL1A1",
    "CTHRC1",
    "POSTN",
    "SFTPC",
    "AGER"
]

genes = [g for g in genes if g in adata.var_names]
genes

In [ ]:
for gene in genes:
    sc.pl.spatial(
        adata_sample,
        color=gene,
        library_id=sample,
        spot_size=1.3,
        title=gene
    )

## Compact AnnData architecture summary

In [ ]:
print("AnnData structure")
print("------------------")
print("X:", adata.shape)
print("obs:", adata.obs.shape)
print("var:", adata.var.shape)

print("\nSelected obsm entries:")
for key in adata.obsm.keys():
    try:
        print(f"{key}: {adata.obsm[key].shape}")
    except Exception:
        pass

## Summary

This notebook establishes the processed Visium dataset used throughout the project. The object contains:

- spatially resolved gene expression,
- integrated scVI embeddings,
- cell2location-derived abundance estimates for 37 cell populations,
- pathway activity scores,
- precomputed NMF factors,
- NMF-defined tissue niche labels,
- and spatial image/coordinate metadata.

The next notebook, `02_cell2location_nmf_niches.ipynb`, focuses on the cell2location abundance matrix and the NMF-defined multicellular tissue niches.